In [17]:
import matplotlib
# 恢复使用非交互式后端，规避 matplotlib-inline 的版本冲突 Bug
matplotlib.use('Agg')  

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, to_tree, fcluster, leaves_list
from matplotlib import rcParams
import os

# 引入 IPython 的显示模块用于渲染最终图片
from IPython.display import Image, display

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'PingFang SC', 'Microsoft YaHei', 'sans-serif']
rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 900

CLUSTER_COLORS = ['#E6B0AA', '#A9CCE3', '#A3E4D7', '#D7BDE2', '#FADBD8', '#F9E795', '#A2D9CE', '#AED6F1']

def _draw_circular_dendrogram_node(ax, node, angle_map, max_dist, cluster_assignments, current_cluster_color_map, color_threshold):
    left_child, right_child = node.get_left(), node.get_right()
    current_radius = max_dist - node.dist
    
    node_is_in_cluster = node.dist <= color_threshold 
    if node_is_in_cluster and node.get_id() in cluster_assignments:
        line_color = current_cluster_color_map[cluster_assignments[node.get_id()]]
    else:
        line_color = '#777777' 
        
    if not left_child and not right_child: return
    
    left_angle, right_angle = angle_map[left_child.get_id()], angle_map[right_child.get_id()]
    if right_angle < left_angle: left_angle, right_angle = right_angle, left_angle
    if right_angle - left_angle > np.pi: left_angle, right_angle = right_angle, left_angle + 2*np.pi
    
    arc_angles = np.linspace(left_angle, right_angle, 100)
    ax.plot(arc_angles, [current_radius] * 100, color=line_color, lw=1.5, zorder=1)
    ax.plot([left_angle, left_angle], [current_radius, max_dist - left_child.dist], color=line_color, lw=1.5, zorder=1)
    ax.plot([right_angle, right_angle], [current_radius, max_dist - right_child.dist], color=line_color, lw=1.5, zorder=1)
    
    if left_child: _draw_circular_dendrogram_node(ax, left_child, angle_map, max_dist, cluster_assignments, current_cluster_color_map, color_threshold)
    if right_child: _draw_circular_dendrogram_node(ax, right_child, angle_map, max_dist, cluster_assignments, current_cluster_color_map, color_threshold)

def calculate_eer_threshold(features, labels):
    scaled_dists = pdist(features, metric='euclidean')
    
    n = len(features)
    iu1 = np.triu_indices(n, 1)
    same_tiger = (labels[:, None] == labels[None, :])
    
    intra_dists = scaled_dists[same_tiger[iu1]]
    inter_dists = scaled_dists[~same_tiger[iu1]]
    
    thresholds = np.linspace(scaled_dists.min(), scaled_dists.max(), 1000)
    fars = []
    frrs = []
    
    for t in thresholds:
        far = np.sum(inter_dists <= t) / len(inter_dists)
        frr = np.sum(intra_dists > t) / len(intra_dists)
        fars.append(far)
        frrs.append(frr)
        
    fars = np.array(fars)
    frrs = np.array(frrs)
    
    eer_idx = np.argmin(np.abs(fars - frrs))
    eer_threshold = thresholds[eer_idx]
    
    print(f"统计分析: 错误接受率和错误拒绝率的等错误率交点为 错误接受率={fars[eer_idx]:.4f}, 错误拒绝率={frrs[eer_idx]:.4f}")
    return eer_threshold

def draw_figC():
    if not os.path.exists("data_BC.npz"):
        print("错误: 找不到 data_BC.npz")
        return

    data = np.load("data_BC.npz")
    features = data['features']
    anonymous_ids = data['ids']
    true_labels = data['labels'] 

    Z = linkage(features, method='ward')
    
    dynamic_threshold = calculate_eer_threshold(features, true_labels)
    print(f"基于严谨的等错误率算法确定的判定阈值为: {dynamic_threshold:.2f}")
    
    tree = to_tree(Z)
    max_dist = tree.dist
    
    clusters = fcluster(Z, dynamic_threshold, criterion='distance')
    unique_clusters = sorted(np.unique(clusters))
    
    print(f"在阈值 T={dynamic_threshold:.2f} 下，系统实际发现了 {len(unique_clusters)} 个聚类簇。")

    cluster_color_map = {cluster_id: CLUSTER_COLORS[i % len(CLUSTER_COLORS)] for i, cluster_id in enumerate(unique_clusters)}
    cluster_assignments = {i: cluster_id for i, cluster_id in enumerate(clusters)}
    
    num_leaves = len(features)
    for i in range(len(Z)):
        node_id = num_leaves + i
        left_child_id, right_child_id = int(Z[i,0]), int(Z[i,1])
        if left_child_id in cluster_assignments and right_child_id in cluster_assignments and \
            cluster_assignments[left_child_id] == cluster_assignments[right_child_id]:
            cluster_assignments[node_id] = cluster_assignments[left_child_id]

    leaf_order = leaves_list(Z)
    angles = np.linspace(0, 2 * np.pi, num_leaves, endpoint=False)

    ordered_labels = [str(anonymous_ids[i]) for i in leaf_order]

    fig, ax = plt.subplots(figsize=(14, 14), subplot_kw={'projection': 'polar'})
    ax.set_facecolor('white')
    fig.patch.set_facecolor('white')

    label_radial_pos = max_dist * 1.02 

    for i in range(num_leaves):
        angle = angles[i]
        label = ordered_labels[i]
        
        rotation_degrees = np.degrees(angle)
        if 90 < rotation_degrees < 270:
            rotation_degrees += 180
            ha_align = 'right'
        else:
            ha_align = 'left'
            
        ax.text(angle, label_radial_pos, label, rotation=rotation_degrees, 
                ha=ha_align, va='center', rotation_mode='anchor',
                fontsize=11, color='#111111', fontweight='bold')

    angle_map = {leaf_order[i]: angles[i] for i in range(num_leaves)}
    for i in range(len(Z)):
        node_id = num_leaves + i
        left_child, right_child = int(Z[i, 0]), int(Z[i, 1])
        angle_map[node_id] = np.arctan2(
            np.sin(angle_map[left_child]) + np.sin(angle_map[right_child]),
            np.cos(angle_map[left_child]) + np.cos(angle_map[right_child])
        )

    _draw_circular_dendrogram_node(ax, tree, angle_map, max_dist, cluster_assignments, cluster_color_map, dynamic_threshold)
    
    threshold_radius = max_dist - dynamic_threshold
    ax.plot(np.linspace(0, 2*np.pi, 200), [threshold_radius]*200, c='#DC0000', ls='--', lw=2.5, zorder=5)
    
    ax.text(np.pi/4, threshold_radius + (max_dist*0.02), f'Treshold (T={dynamic_threshold:.2f})', 
            c='#DC0000', ha='left', va='bottom', fontsize=16, fontweight='bold', zorder=6)
    
    ax.plot(0, 0, 'o', markersize=8, color='#555555', zorder=2)
    
    ax.set_ylim(0, max_dist * 1.1)
    ax.set_yticklabels([]); ax.set_xticklabels([])
    ax.spines['polar'].set_visible(False); ax.grid(False)

    plt.tight_layout()
    
    output_filename = "Fig2C_Circular_Dendrogram.png"
    plt.savefig(output_filename, bbox_inches='tight')
    plt.close(fig) 
    print("✅ 图 2C 保存成功！")

    display(Image(filename=output_filename))

if __name__ == "__main__":
    draw_figC()

统计分析: 错误接受率和错误拒绝率的等错误率交点为 错误接受率=0.0215, 错误拒绝率=0.0195
基于严谨的等错误率算法确定的判定阈值为: 59.34
在阈值 T=59.34 下，系统实际发现了 31 个聚类簇。
✅ 图 2C 保存成功！
